# Phase 10.7 — Learning-to-Rank

Train a reference LightGBM LambdaMART model on a small fixture dataset (production baseline unchanged).

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "productiq").is_dir():
    ROOT = ROOT.parents[1]
sys.path.insert(0, str(ROOT / "src"))

from productiq.ranking.dataset_builder import RankingDatasetQueryGroup, build_ranking_dataset_from_query_groups
from productiq.ranking.feature_schema import RankingFeatures
from productiq.ranking.ltr import (
    LTRTrainingConfig,
    ORDERED_LTR_FEATURE_NAMES,
    build_feature_matrix_from_dataset,
    split_ranking_dataset_by_query,
    train_ltr_model,
    load_ltr_model_artifact,
    predict_ltr_scores,
)
from productiq.ranking.ltr.config import LTRQuerySplitConfig

In [ ]:
def feature_row(product_id: str) -> RankingFeatures:
    return RankingFeatures(
        product_id=product_id,
        retrieval={
            "bm25_score": 1.0,
            "retrieved_by_bm25": True,
            "retrieved_by_vector": False,
            "retrieved_by_both": False,
        },
        matching={"matched_attribute_count": 0, "brand_match": True},
        catalog={"discount_price_inr": 100, "original_price_inr": 200, "discount_amount_inr": 100},
    )


def group(qid: str, pids: tuple[str, ...], rel: set[str]) -> RankingDatasetQueryGroup:
    return RankingDatasetQueryGroup(
        query_id=qid,
        features=tuple(feature_row(pid) for pid in pids),
        relevance_label_by_product_id={pid: (1 if pid in rel else 0) for pid in pids},
    )


dataset = build_ranking_dataset_from_query_groups(
    (
        group("Q1", ("P1", "P2", "P3"), {"P1"}),
        group("Q2", ("P4", "P5"), {"P4"}),
        group("Q3", ("P6", "P7", "P8"), {"P6"}),
        group("Q4", ("P9", "P10"), {"P10"}),
    ),
    query_set_version="notebook-demo",
    judgment_source_version="manual-demo",
    require_labels=True,
)
split, _ = split_ranking_dataset_by_query(dataset, config=LTRQuerySplitConfig(split_seed=42))
matrix = build_feature_matrix_from_dataset(dataset)
{
    "rows": len(dataset.rows),
    "features": len(ORDERED_LTR_FEATURE_NAMES),
    "split": split.model_dump(),
    "matrix_shape": matrix.matrix.shape,
}

In [ ]:
artifact_dir = ROOT / "resources" / "models" / "ranking_ltr_reference_v10_7_0"
config = LTRTrainingConfig(split=LTRQuerySplitConfig(split_seed=42))
result, trained = train_ltr_model(dataset, config, artifact_directory=artifact_dir)
loaded = load_ltr_model_artifact(artifact_dir)
scores = predict_ltr_scores(loaded, matrix)
{
    "artifact_dir": str(artifact_dir),
    "test_predictions": len(result.test_predictions),
    "score_preview": scores[:5].tolist(),
}